# Lecture 3 Feature Selection

In [11]:
import pandas as pd
import yfinance as yf
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as ss
from sklearn import linear_model
import statsmodels.api as sm

In [14]:
# Figure configuration
plt.rcParams['figure.figsize'] = (16, 6)

# define dataset
start_date = datetime(2020,1,1)
end_date = datetime(2023,12,31)
DATA = pd.read_csv('INFO6105_FeatureMart.csv')  # 不设置 index_col
DATA['Date'] = pd.to_datetime(DATA['Date'])  # 转换 Date 列为 datetime 格式

# X and Y have to use the same index and index must be of the same data type

STOCK = yf.download('MSFT', start_date, end_date)
STOCK.describe()
X = DATA.set_index('Date').fillna(method='bfill')
# y = STOCK['Adj Close']
y = np.diff(np.log(STOCK['Adj Close'].values))
y = np.append(y[0], y)

[*********************100%***********************]  1 of 1 completed
/var/folders/_s/3jlv6djs67d5hnxcsy9mp4y00000gn/T/ipykernel_25505/3513548572.py:14: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  X = DATA.set_index('Date').fillna(method='bfill')


### Benchmark Model: Ordinary Linear Regression

In [15]:
# Benchmark Model: Two-Step Factor Selection (Hard Thresholding Method)
# Motivation: the method selects features according to their significance in 
# an OLS regression for Y
# Step 1: regress Y on each feature in X and remove those whose p-val > 0.05 (|t|<1.96)
# Step 2: regress Y on the subset of X
X = sm.add_constant(X)
benchmark_prep = sm.OLS(y,X).fit()
benchmark_prep.summary()
benchmark_select = X.columns[np.abs(benchmark_prep.tvalues)>=1.96]
x = X[benchmark_select]
benchmark = sm.OLS(y,x).fit()
print(benchmark.summary())
y_hat_benchmark1 = benchmark.predict(x)
corr_benchmark1 = ss.pearsonr(y_hat_benchmark1, y)[0]
print('benchmark: corr (Y, Y_pred) = '+str(corr_benchmark1))
print('Hard Thresholding selected ' +str(len(benchmark_select)) +' features: ', benchmark_select.values)

                                 OLS Regression Results                                
Dep. Variable:                      y   R-squared (uncentered):                   0.794
Model:                            OLS   Adj. R-squared (uncentered):              0.792
Method:                 Least Squares   F-statistic:                              769.4
Date:                Tue, 10 Dec 2024   Prob (F-statistic):                        0.00
Time:                        17:48:25   Log-Likelihood:                          3273.9
No. Observations:                1006   AIC:                                     -6538.
Df Residuals:                    1001   BIC:                                     -6513.
Df Model:                           5                                                  
Covariance Type:            nonrobust                                                  
                 coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------

### Ridge Regression

In [16]:
# Motivation: the model sets the objective function to be the 
# sum of squared residuals from OLS + penalty function 
# that penalizes squared values of beta. A minimization process squeezes 
# the small beta values close to 0. 
# a represents the obj function's sensitivity to the penalty term   
# Steo 1. run Ridge Regression and obtain coefficients
# Step 2. remove features with coefficients close to 0 and run OLS 
a = 0.5
model2_prep = linear_model.Ridge(alpha=a, fit_intercept=False).fit(X, y)
model2_select = X.columns[np.abs(model2_prep.coef_)>=0.001]
x = X[model2_select]
model2 = sm.OLS(y,x).fit()
print(model2.summary())
y_pred_model2 = model2.predict(x)
corr_model2 = ss.pearsonr(y_pred_model2, y)[0]
print('model 2 Ridge Regression: corr (Y, Y_pred) = '+str(corr_model2))
print('Ridge Regression selected ' +str(len(model2_select)) +' features: ', model2_select.values)

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.797
Model:                            OLS   Adj. R-squared:                  0.793
Method:                 Least Squares   F-statistic:                     227.8
Date:                Tue, 10 Dec 2024   Prob (F-statistic):               0.00
Time:                        17:48:27   Log-Likelihood:                 3282.7
No. Observations:                1006   AIC:                            -6529.
Df Residuals:                     988   BIC:                            -6441.
Df Model:                          17                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const              0.0127      0.017      0.

### LASSO

In [6]:
a = 0.5
model3_prep = linear_model.Lasso(alpha=a, fit_intercept=False).fit(X, y)
model3_select = X.columns[np.abs(model3_prep.coef_)!=0.0]
x = X[model3_select]
model3 = sm.OLS(y,x).fit()
print(model3.summary())
y_pred_model3 = model3.predict(x)
corr_model3 = ss.pearsonr(y_pred_model3, y)[0]
print('model 3 LASSO: corr (Y, Y_pred) = '+str(corr_model3))
print('LASSO selected ' +str(len(model3_select)) +' features: ', model3_select.values)

                                 OLS Regression Results                                
Dep. Variable:                      y   R-squared (uncentered):                   0.002
Model:                            OLS   Adj. R-squared (uncentered):             -0.000
Method:                 Least Squares   F-statistic:                             0.9744
Date:                Sun, 08 Dec 2024   Prob (F-statistic):                       0.378
Time:                        16:49:17   Log-Likelihood:                          2481.4
No. Observations:                1006   AIC:                                     -4959.
Df Residuals:                    1004   BIC:                                     -4949.
Df Model:                           2                                                  
Covariance Type:            nonrobust                                                  
                 coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------

### Elastic Net

In [7]:
a = 0.5
model4_prep = linear_model.ElasticNet(alpha=a, fit_intercept=False).fit(X, y)
model4_select = X.columns[np.abs(model4_prep.coef_)!=0.0]
x = X[model4_select]
model4 = sm.OLS(y,x).fit()
print(model4.summary())
y_pred_model4 = model4.predict(x)
corr_model4 = ss.pearsonr(y_pred_model4, y)[0]
print('model 4 Elastic Net: corr (Y, Y_pred) = '+str(corr_model3))
print('ElasticNet selected ' +str(len(model4_select)) +' features: ', model4_select.values)

                                 OLS Regression Results                                
Dep. Variable:                      y   R-squared (uncentered):                   0.003
Model:                            OLS   Adj. R-squared (uncentered):             -0.000
Method:                 Least Squares   F-statistic:                             0.9700
Date:                Sun, 08 Dec 2024   Prob (F-statistic):                       0.406
Time:                        16:49:21   Log-Likelihood:                          2481.8
No. Observations:                1006   AIC:                                     -4958.
Df Residuals:                    1003   BIC:                                     -4943.
Df Model:                           3                                                  
Covariance Type:            nonrobust                                                  
                 coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------

### Least Angle Regression

In [8]:
# Motivation: the model adds features one by one (stepwise)
# it selects features by looking for feature X[n+1] that has the highest 
# correlation with residuals obtained from regressing Y on all pre-selected Xs.  
# Step 1. run Least Angle Regression (LARS) and obtain coefficients
# Step 2. select features with non-0 coefficients alone and run OLS 
model1_prep = linear_model.Lars().fit(X, y)
model1_select = X.columns[model1_prep.coef_>=0.001]
x = X[model1_select]
model1 = sm.OLS(y,x).fit()
print(model1.summary())
y_pred_model1 = model1.predict(x)
corr_model1 = ss.pearsonr(y_pred_model1, y)[0]
print('model 1 LARS: corr (Y, Y_pred) = '+str(corr_model1))
print('LARS selected ' +str(len(model1_select)) +' features: ', model1_select.values + '\n')

                                 OLS Regression Results                                
Dep. Variable:                      y   R-squared (uncentered):                   0.721
Model:                            OLS   Adj. R-squared (uncentered):              0.718
Method:                 Least Squares   F-statistic:                              183.5
Date:                Sun, 08 Dec 2024   Prob (F-statistic):                   1.47e-263
Time:                        16:49:25   Log-Likelihood:                          3123.3
No. Observations:                1006   AIC:                                     -6219.
Df Residuals:                     992   BIC:                                     -6150.
Df Model:                          14                                                  
Covariance Type:            nonrobust                                                  
                     coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------